# Notebook 01 — Data Understanding and Problem Framing
**Project:** Mining Process Quality Prediction — Minsur Analytics Challenge  
**Level:** 1 (mandatory)  
**Target:** `% Silica Concentrate`  
**Author:** Analytics Candidate  

---

## Business Context

In the Minsur flotation process, the quality of the final iron ore concentrate
is measured by the **percentage of silica** (`% Silica Concentrate`) — a lower value
means purer iron concentrate. Laboratory quality measurements arrive with a
**significant delay** (up to several hours), which prevents real-time operational
decisions.

The goal of this project is to **predict % Silica Concentrate in real time**
using continuous sensor data (ore feed quality, reagent flows, flotation column
parameters), enabling operators to react proactively to quality excursions.

---

## Why Temporal Integrity Matters

> A random train/test split on time-series data introduces **data leakage**:
> the model would train on observations from March to predict January, effectively
> seeing the future. Evaluation metrics would be over-optimistic and the model
> would fail in production. We strictly respect the chronological ordering of
> all splits throughout this project.

In [ ]:
# ---------------------------------------------------------------------------
# 0. Environment setup — add project root to sys.path so src/ is importable
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.config import CFG, PROJECT_ROOT
from src.data_preprocessing import (
    load_raw_data,
    normalize_column_names,
    parse_and_sort_datetime,
    remove_duplicates,
    data_quality_report,
    outlier_summary,
    sampling_frequency_report,
    run_preprocessing_pipeline,
)
from src.evaluate import compute_metrics

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
FIGURES_PATH = Path(CFG['paths']['reports_figures'])

TARGET = CFG['target']
print(f"Target: {TARGET}")

---
## 1. Load Dataset

The raw CSV is expected in `data/raw/`. If not yet downloaded, run the Kaggle
download cell below (requires `kagglehub` and valid Kaggle credentials).

In [ ]:
# ---------------------------------------------------------------------------
# Download via kagglehub if the file does not already exist
# ---------------------------------------------------------------------------
raw_dir = Path(CFG['paths']['data_raw'])
raw_file = raw_dir / CFG['data']['raw_filename']

if not raw_file.exists():
    print('Downloading dataset from Kaggle...')
    import kagglehub
    from kagglehub import KaggleDatasetAdapter
    df_kaggle = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        'edumagalhaes/quality-prediction-in-a-mining-process',
        '',
    )
    df_kaggle.to_csv(raw_file, index=False)
    print(f'Saved to: {raw_file}')
else:
    print(f'File already exists: {raw_file}')

In [ ]:
# Load raw (uncleaned) to inspect as-is
df_raw = load_raw_data(CFG)
df_raw = normalize_column_names(df_raw)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

---
## 2. Datetime Parsing and Temporal Ordering

In [ ]:
df = parse_and_sort_datetime(df_raw.copy(), CFG)
df = remove_duplicates(df)

print(f'Date range: {df.index.min()} → {df.index.max()}')
print(f'Shape after deduplication: {df.shape}')
df.head(3)

---
## 3. Target and Feature Identification

In [ ]:
# Identify target and candidate predictors
lab_cols = CFG['feature_groups']['lab_measurements']  # delayed — NOT predictors

feature_candidates = [c for c in df.columns if c not in lab_cols]

print(f'Target: {TARGET}')
print(f'\nLab measurement columns (delayed — excluded from features):')
print(lab_cols)
print(f'\nCandidate predictors ({len(feature_candidates)}):')
for c in feature_candidates:
    print(f'  {c}')

> **Design decision — Leakage guard:**  
> `% Iron Concentrate` and `% Silica Concentrate` are laboratory measurements
> that are updated only once per hour (or less frequently) and always arrive
> **after** the sensor readings they correspond to. Using them directly as
> features would be leakage — at inference time they are not yet available.
> Only **lagged** versions (shift ≥ 1) are acceptable, and only when documented.

---
## 4. Data Quality Analysis

In [ ]:
report = data_quality_report(df)
print('=== Data Quality Report ===')
report

In [ ]:
# Missing values summary
missing = report[['n_missing', 'pct_missing']].sort_values('n_missing', ascending=False)
print('Missing values per column:')
print(missing[missing['n_missing'] > 0])
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Outlier summary (IQR × 3)
out_summary = outlier_summary(df)
print('Outlier summary (IQR × 3 factor):')
print(out_summary.head(15))

In [ ]:
# Sampling frequency analysis
freq_report = sampling_frequency_report(df)
print('Inter-sample time deltas (minutes):')
print(freq_report.head(10))

### Mixed Sampling Frequencies — Implications

The dataset contains variables with **different update frequencies**:

| Variable group | Approx. update frequency |
|---|---|
| Sensor data (flows, levels, pH, density) | Every ~20 minutes |
| Lab measurements (`% Iron`, `% Silica Concentrate`) | Every ~1 hour (with delay) |

**Implications:**
1. **Repeated values**: Lab columns hold the same value for multiple consecutive rows, creating
   artificially high autocorrelation and potentially misleading correlation statistics.
2. **Leakage risk**: Because lab readings are recorded at irregular intervals, any model that
   uses the current lab value as a feature would be using a value that was not yet available
   at the time the sensor reading was taken.
3. **Rolling feature design**: When computing rolling statistics, we must ensure the window
   size is chosen in physical time, not just in row count, since rows are not equally spaced.
4. **Target smoothing**: The target variable itself (`% Silica Concentrate`) may appear
   'frozen' for several consecutive rows. This introduces autocorrelation that models must
   not exploit in a leaky way.

---
## 5. Visual Analysis

In [ ]:
# ── 5a. Target distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df[TARGET].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Distribution of {TARGET}', fontsize=12)
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel('Count')

axes[1].boxplot(df[TARGET].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title(f'Boxplot — {TARGET}', fontsize=12)
axes[1].set_ylabel(TARGET)
axes[1].set_xticks([])

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(df[TARGET].describe().round(4))

In [ ]:
# ── 5b. Temporal evolution of the target ─────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df[TARGET], lw=0.5, color='steelblue', alpha=0.7)
ax.set_title(f'Temporal Evolution — {TARGET}', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel(TARGET)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'target_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5c. Correlation matrix ────────────────────────────────────────────────
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

# Focus on top correlated features with target
target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
top_cols = target_corr.head(15).index.tolist() + [TARGET]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr.loc[top_cols, top_cols], dtype=bool))
sns.heatmap(
    corr.loc[top_cols, top_cols],
    annot=True, fmt='.2f', mask=mask, cmap='RdBu_r',
    center=0, linewidths=0.5, ax=ax, annot_kws={'fontsize': 8}
)
ax.set_title(f'Correlation Matrix — Top features vs {TARGET}', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 correlations with target:')
print(target_corr.head(10).round(3))

In [ ]:
# ── 5d. Missing values heatmap ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
missing_pct = df.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0]

if len(missing_pct) > 0:
    missing_pct.sort_values(ascending=False).plot(kind='bar', color='tomato', ax=ax)
    ax.set_title('Missing Values (%) by Column', fontsize=12)
    ax.set_ylabel('% Missing')
    ax.set_xlabel('')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / 'missing_values.png', dpi=150, bbox_inches='tight')
else:
    ax.text(0.5, 0.5, 'No missing values found', ha='center', va='center',
            transform=ax.transAxes, fontsize=14)
    ax.set_title('Missing Values', fontsize=12)

plt.show()

---
## 6. Metric Definitions and Baseline Model

### Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error in % Silica units |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Penalises large errors; relevant for quality excursions |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of variance explained; 1.0 = perfect |

### Baseline Strategy
The simplest possible model: **predict the mean of the training target for every observation**.  
Any useful model must beat this baseline; otherwise it provides no value over the current
status quo of 'just use the average'.

In [ ]:
# ---------------------------------------------------------------------------
# Temporal split for baseline evaluation — NO random split allowed
# ---------------------------------------------------------------------------
from src.feature_engineering import temporal_split

# Use the cleaned full DataFrame (no feature engineering yet)
df_clean = df.dropna(subset=[TARGET])

train_df, val_df, test_df = temporal_split(
    df_clean,
    train_ratio=CFG['split']['train_ratio'],
    val_ratio=CFG['split']['val_ratio'],
)

In [ ]:
# Baseline: predict training mean on all splits
train_mean = train_df[TARGET].mean()

baseline_results = {}
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    y_true = split_df[TARGET]
    y_pred = np.full(len(y_true), train_mean)
    metrics = compute_metrics(y_true, y_pred)
    baseline_results[split_name] = metrics
    print(f'Baseline ({split_name}): {metrics}')

print(f'\nBaseline prediction (mean of training target): {train_mean:.4f}')

In [ ]:
# Save baseline metrics for comparison in notebook 02
import json
metrics_path = Path(CFG['paths']['reports_metrics'])
with open(metrics_path / 'baseline_metrics.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print(f'Baseline metrics saved to: {metrics_path / "baseline_metrics.json"}')

---
## 7. Save Cleaned Data for Downstream Notebooks

In [ ]:
# Run the full preprocessing pipeline and persist to data/interim/
df_cleaned = run_preprocessing_pipeline(CFG)
print(f'Cleaned data shape: {df_cleaned.shape}')
df_cleaned.head(3)

---
## Summary — Level 1 Checklist

| Item | Status |
|---|---|
| Dataset loaded from `data/raw/` | ✅ |
| Datetime column parsed and sorted | ✅ |
| Target identified: `% Silica Concentrate` | ✅ |
| Candidate predictors identified | ✅ |
| Missing values analysed | ✅ |
| Duplicates removed | ✅ |
| Outliers flagged (IQR × 3) | ✅ |
| Mixed sampling frequencies explained | ✅ |
| Visual analysis (distribution, temporal, correlation, missing) | ✅ |
| Metric definitions (MAE, RMSE, R²) | ✅ |
| Baseline model implemented and evaluated | ✅ |
| No random split — temporal integrity preserved | ✅ |
| No data leakage | ✅ |